In [1]:
import glob
import re

import numpy as np
import pandas as pd
from translate2pdbqt import cif_to_pdbqt, pdb_to_pdbqt, smiles_to_pdbqt

## Translate the Receptor file
First the prepared receptor needs to be translated to a file format that we can dock with. Provide either a `.cif` or `.pdb` file.

In [2]:
input_file = "HIV1protease.pdb"
base, extension = input_file.split(".")
pdbqt_file1 = base + ".pdbqt"

if extension == "cif":
    cif_to_pdbqt(input_file, pdbqt_file1)
elif extension == "pdb":
    pdb_to_pdbqt(input_file, pdbqt_file1)
else:
    print("Please provide either .cif or .pdb files!")

## Translate the Ligand files
Now the ligand `SMILES` are translated as well. Provide either a `.csv` file with a column called `smiles`.

In [3]:
ligand_data_file = "HIV_ligands.csv"
ligand_data = pd.read_csv(ligand_data_file)
smiles = ligand_data.smiles.to_list()
print(f"{len(smiles)} SMILES extracted.")

4 SMILES extracted.


In [4]:
pdbqts = [
    smiles_to_pdbqt(smi, str(f"ligand_{i}.pdbqt")) for i, smi in enumerate(smiles)
]
print(f"{len(pdbqts)} SMILES translated successfully to PDBQT.")

4 SMILES translated successfully to PDBQT.


## Docking
Now we can start the docking process, by executing the cell below:

In [5]:
for ligand in pdbqts:
    name = ligand.removesuffix(".pdbqt")
    ! /mnt/sds-hd/sd25g005/docking/bin/vina --receptor "$pdbqt_file1" --ligand "$ligand" \
           --config box_dimensions.txt \
           --exhaustiveness=32 --out "$base"_"$name"_vina_out.pdbqt > "$base"_"$name"_vina.log
    print(f"Finished docking {ligand}.")

Finished docking ligand_0.pdbqt.
Finished docking ligand_1.pdbqt.
Finished docking ligand_2.pdbqt.
Finished docking ligand_3.pdbqt.


Now we can collect the results by looking for the affinity valies in the Vina `.log` files:

In [ ]:
digits = []
affinities = []

for f in sorted(glob.glob(f"{base}_ligand_*_vina.log")):
    digits.append(re.search(r"_(\d+)_", f).group(1))
    with open(f, "r") as fh:
        text = fh.read()
    m = re.search(r"^\s*(\d+)\s+(-?\d+\.\d+)\s+(\d+)\s+(\d+)\s*$", text, re.MULTILINE)
    if m:
        affinities.append(m.groups()[1])
    else:
        affinities.append(np.nan)

vina_results = pd.DataFrame(
    {"Vina affinity": pd.to_numeric(affinities)}, index=pd.to_numeric(digits)
).sort_index()

print("5 best candidates:")
vina_results.sort_values(by="Vina affinity").head()

5 best candidates:


,Vina affinity
1,-10.940
0,-10.530
2,-9.303
3,-8.613


We can also add them to the initial table you provided:

In [7]:
ligand_data = ligand_data.join(vina_results, how="inner")
ligand_data.to_csv(f"{base}_{ligand_data_file}")